In [183]:
import pandas as pd
import numpy as np

In [184]:
#Merging all three into one dataset

housing = pd.read_csv(r"C:\Users\prith\Downloads\airbnb-data-pipeline\airbnb-data-pipeline\data\processed\listing_clean.csv")
reviews = pd.read_csv(r"C:\Users\prith\Downloads\airbnb-data-pipeline\airbnb-data-pipeline\data\processed\reviews_clean.csv")
calendar = pd.read_csv(r"C:\Users\prith\Downloads\airbnb-data-pipeline\airbnb-data-pipeline\data\processed\calendar_clean.csv")

housing = housing.merge(reviews, left_on = 'id', right_on = 'listing_id', how='left')
housing = housing.merge(calendar, left_on = 'id', right_on = 'listing_id', how='left')

In [185]:
print(housing.head(10).to_string())

      id  host_id  hosts_time_as_user_years  hosts_time_as_user_months  hosts_time_as_host_years  hosts_time_as_host_months  host_is_superhost  host_listings_count neighbourhood_cleansed neighbourhood_group_cleansed   latitude  longitude                property_type        room_type  accommodates  bathrooms  bedrooms   price  minimum_nights  maximum_nights  minimum_nights_avg_ntm  maximum_nights_avg_ntm  has_availability  availability_30  availability_60  availability_90  availability_365  number_of_reviews  number_of_reviews_ltm  number_of_reviews_l30d  availability_eoy  number_of_reviews_ly  estimated_occupancy_l365d  days_since_first_review  days_since_last_review  has_reviews  review_scores_rating  review_scores_accuracy  review_scores_cleanliness  review_scores_checkin  review_scores_communication  review_scores_location  review_scores_value  calculated_host_listings_count  calculated_host_listings_count_entire_homes  calculated_host_listings_count_private_rooms  calculated_host_l

In [186]:
print(housing.dtypes.to_string())

id                                                int64
host_id                                           int64
hosts_time_as_user_years                        float64
hosts_time_as_user_months                       float64
hosts_time_as_host_years                        float64
hosts_time_as_host_months                       float64
host_is_superhost                                  bool
host_listings_count                             float64
neighbourhood_cleansed                              str
neighbourhood_group_cleansed                        str
latitude                                        float64
longitude                                       float64
property_type                                       str
room_type                                           str
accommodates                                      int64
bathrooms                                       float64
bedrooms                                        float64
price                                           

In [187]:
print(housing.shape)

(20680, 69)


In [188]:
print((housing.isnull().mean()*100).to_string())

id                                               0.000000
host_id                                          0.000000
hosts_time_as_user_years                         0.000000
hosts_time_as_user_months                        0.000000
hosts_time_as_host_years                         0.000000
hosts_time_as_host_months                        0.000000
host_is_superhost                                0.000000
host_listings_count                              0.000000
neighbourhood_cleansed                           0.000000
neighbourhood_group_cleansed                     0.000000
latitude                                         0.000000
longitude                                        0.000000
property_type                                    0.000000
room_type                                        0.000000
accommodates                                     0.000000
bathrooms                                        0.000000
bedrooms                                         0.000000
price         

In [189]:
#filling the null values in few_columns
housing['review_count'] = housing['review_count'].fillna(0)
housing['avg_comment_length'] = housing['avg_comment_length'].fillna(0)

housing = housing.drop(columns=['listing_id_x', 'listing_id_y']) #duplicate listing_id

# droping the date columns from reviews as we already have days_since in listings
housing = housing.drop(columns=['first_review_date', 'last_review_date'])


In [190]:
print((housing.isnull().mean()*100).to_string())

id                                              0.0
host_id                                         0.0
hosts_time_as_user_years                        0.0
hosts_time_as_user_months                       0.0
hosts_time_as_host_years                        0.0
hosts_time_as_host_months                       0.0
host_is_superhost                               0.0
host_listings_count                             0.0
neighbourhood_cleansed                          0.0
neighbourhood_group_cleansed                    0.0
latitude                                        0.0
longitude                                       0.0
property_type                                   0.0
room_type                                       0.0
accommodates                                    0.0
bathrooms                                       0.0
bedrooms                                        0.0
price                                           0.0
minimum_nights                                  0.0
maximum_nigh

In [191]:
housing.shape

(20680, 65)

In [192]:
print(housing.head(10).to_string())

      id  host_id  hosts_time_as_user_years  hosts_time_as_user_months  hosts_time_as_host_years  hosts_time_as_host_months  host_is_superhost  host_listings_count neighbourhood_cleansed neighbourhood_group_cleansed   latitude  longitude                property_type        room_type  accommodates  bathrooms  bedrooms   price  minimum_nights  maximum_nights  minimum_nights_avg_ntm  maximum_nights_avg_ntm  has_availability  availability_30  availability_60  availability_90  availability_365  number_of_reviews  number_of_reviews_ltm  number_of_reviews_l30d  availability_eoy  number_of_reviews_ly  estimated_occupancy_l365d  days_since_first_review  days_since_last_review  has_reviews  review_scores_rating  review_scores_accuracy  review_scores_cleanliness  review_scores_checkin  review_scores_communication  review_scores_location  review_scores_value  calculated_host_listings_count  calculated_host_listings_count_entire_homes  calculated_host_listings_count_private_rooms  calculated_host_l

In [193]:
#doing the feature engineering (creating new attributes):

housing['price_per_person'] = housing['price']/housing['accommodates']

housing['price_per_bedroom'] = housing['price']/housing['bedrooms'].replace(0,1)

housing['log_price'] = np.log1p(housing['price'])

housing['host_experience_level'] = pd.cut(
    housing['hosts_time_as_host_years'],
    bins= [-1, 1, 3, 9999],
    labels= ['New', 'Intermediate', 'Experienced']
)

housing['avail_0_30'] = housing['availability_30'] / 30
housing['avail_31_60'] = (housing['availability_60'] - housing['availability_30']) / 30
housing['avail_61_90'] = (housing['availability_90'] - housing['availability_60']) / 30
housing['avail_91_365'] = (housing['availability_365'] - housing['availability_90']) / 275

housing['total_availability_score'] = (
    housing['avail_0_30'] * 0.4 +
    housing['avail_31_60'] * 0.3 +
    housing['avail_61_90'] * 0.2 +
    housing['avail_91_365'] * 0.1
)


In [194]:
housing[['price_per_person', 'price_per_bedroom', 'log_price', 'total_availability_score']].describe()

,price_per_person,price_per_bedroom,log_price,total_availability_score
count,20680.000000,20680.000000,20680.000000,20680.000000
mean,93.704301,192.594206,5.141425,0.619201
std,159.249746,318.546489,0.813969,0.339379
min,2.290000,2.426667,1.699279,0.000364
25%,43.885000,76.570000,4.552086,0.312727
50%,66.570000,132.757500,5.125600,0.689061
75%,105.677500,221.192500,5.636574,0.960000
max,10944.830000,12279.070000,9.620859,1.000000


In [195]:
#adding a few more new attributes
review_cols = ['review_scores_rating', 'review_scores_accuracy', 
               'review_scores_cleanliness', 'review_scores_checkin',
               'review_scores_communication', 'review_scores_location', 
               'review_scores_value']

housing['review_score_avg'] = housing[review_cols].mean(axis=1)

median_reviews = housing['number_of_reviews'].median()
housing['is_high_demand'] = (housing['number_of_reviews'] > median_reviews).astype(int)

In [196]:
housing.shape

(20680, 76)

In [197]:
print(housing.head(10).to_string())

      id  host_id  hosts_time_as_user_years  hosts_time_as_user_months  hosts_time_as_host_years  hosts_time_as_host_months  host_is_superhost  host_listings_count neighbourhood_cleansed neighbourhood_group_cleansed   latitude  longitude                property_type        room_type  accommodates  bathrooms  bedrooms   price  minimum_nights  maximum_nights  minimum_nights_avg_ntm  maximum_nights_avg_ntm  has_availability  availability_30  availability_60  availability_90  availability_365  number_of_reviews  number_of_reviews_ltm  number_of_reviews_l30d  availability_eoy  number_of_reviews_ly  estimated_occupancy_l365d  days_since_first_review  days_since_last_review  has_reviews  review_scores_rating  review_scores_accuracy  review_scores_cleanliness  review_scores_checkin  review_scores_communication  review_scores_location  review_scores_value  calculated_host_listings_count  calculated_host_listings_count_entire_homes  calculated_host_listings_count_private_rooms  calculated_host_l

In [198]:
#dropping few columns that are not much needed
cols_to_drop = [
    'hosts_time_as_user_years',
    'hosts_time_as_user_months',
    'hosts_time_as_host_months',

    'availability_30',
    'availability_60',
    'availability_90',
    'availability_365',
    'avail_0_30',
    'avail_31_60',
    'avail_61_90',
    'avail_91_365',
    'availability_eoy',
    'total_days',           # every row is 365, zero variance

    'number_of_reviews_l30d',
    'number_of_reviews_ly',
    'review_count',         

    'review_scores_rating',
    'review_scores_accuracy',
    'review_scores_checkin',
    'review_scores_communication',
    'review_scores_value',

    'calculated_host_listings_count_entire_homes',
    'calculated_host_listings_count_private_rooms',
    'calculated_host_listings_count_shared_rooms',
    'host_listings_count',  

    'avg_comment_length',   # weak signal, review_score_avg captures quality better
    'reviews_per_month',    
]

housing = housing.drop(columns=cols_to_drop)

In [199]:
print(housing.shape)
print(housing.columns.tolist())

(20680, 49)
['id', 'host_id', 'hosts_time_as_host_years', 'host_is_superhost', 'neighbourhood_cleansed', 'neighbourhood_group_cleansed', 'latitude', 'longitude', 'property_type', 'room_type', 'accommodates', 'bathrooms', 'bedrooms', 'price', 'minimum_nights', 'maximum_nights', 'minimum_nights_avg_ntm', 'maximum_nights_avg_ntm', 'has_availability', 'number_of_reviews', 'number_of_reviews_ltm', 'estimated_occupancy_l365d', 'days_since_first_review', 'days_since_last_review', 'has_reviews', 'review_scores_cleanliness', 'review_scores_location', 'calculated_host_listings_count', 'amenity_count', 'has_wifi', 'has_kitchen', 'has_air_conditioning', 'has_dedicated_workspace', 'has_tv', 'has_heating', 'has_refrigerator', 'has_microwave', 'has_washer', 'has_dishwasher', 'has_elevator', 'has_gym', 'availability_rate', 'price_per_person', 'price_per_bedroom', 'log_price', 'host_experience_level', 'total_availability_score', 'review_score_avg', 'is_high_demand']


In [200]:
#removing the id and host_id
housing = housing.drop(columns=['id', 'host_id'])
print(housing.shape)

(20680, 47)


In [201]:
print(housing.head(10).to_string())

   hosts_time_as_host_years  host_is_superhost neighbourhood_cleansed neighbourhood_group_cleansed   latitude  longitude                property_type        room_type  accommodates  bathrooms  bedrooms   price  minimum_nights  maximum_nights  minimum_nights_avg_ntm  maximum_nights_avg_ntm  has_availability  number_of_reviews  number_of_reviews_ltm  estimated_occupancy_l365d  days_since_first_review  days_since_last_review  has_reviews  review_scores_cleanliness  review_scores_location  calculated_host_listings_count  amenity_count  has_wifi  has_kitchen  has_air_conditioning  has_dedicated_workspace  has_tv  has_heating  has_refrigerator  has_microwave  has_washer  has_dishwasher  has_elevator  has_gym  availability_rate  price_per_person  price_per_bedroom  log_price host_experience_level  total_availability_score  review_score_avg  is_high_demand
0                      15.0               True           Williamsburg                     Brooklyn  40.709350 -73.953420           Entire r

In [202]:
print(housing.dtypes.to_string())

hosts_time_as_host_years           float64
host_is_superhost                     bool
neighbourhood_cleansed                 str
neighbourhood_group_cleansed           str
latitude                           float64
longitude                          float64
property_type                          str
room_type                              str
accommodates                         int64
bathrooms                          float64
bedrooms                           float64
price                              float64
minimum_nights                     float64
maximum_nights                     float64
minimum_nights_avg_ntm             float64
maximum_nights_avg_ntm             float64
has_availability                      bool
number_of_reviews                    int64
number_of_reviews_ltm                int64
estimated_occupancy_l365d            int64
days_since_first_review            float64
days_since_last_review             float64
has_reviews                          int64
review_scor

In [203]:
print(housing.head(10).to_string())

   hosts_time_as_host_years  host_is_superhost neighbourhood_cleansed neighbourhood_group_cleansed   latitude  longitude                property_type        room_type  accommodates  bathrooms  bedrooms   price  minimum_nights  maximum_nights  minimum_nights_avg_ntm  maximum_nights_avg_ntm  has_availability  number_of_reviews  number_of_reviews_ltm  estimated_occupancy_l365d  days_since_first_review  days_since_last_review  has_reviews  review_scores_cleanliness  review_scores_location  calculated_host_listings_count  amenity_count  has_wifi  has_kitchen  has_air_conditioning  has_dedicated_workspace  has_tv  has_heating  has_refrigerator  has_microwave  has_washer  has_dishwasher  has_elevator  has_gym  availability_rate  price_per_person  price_per_bedroom  log_price host_experience_level  total_availability_score  review_score_avg  is_high_demand
0                      15.0               True           Williamsburg                     Brooklyn  40.709350 -73.953420           Entire r

In [204]:
print(housing.describe().to_string())

       hosts_time_as_host_years      latitude     longitude  accommodates     bathrooms      bedrooms         price  minimum_nights  maximum_nights  minimum_nights_avg_ntm  maximum_nights_avg_ntm  number_of_reviews  number_of_reviews_ltm  estimated_occupancy_l365d  days_since_first_review  days_since_last_review   has_reviews  review_scores_cleanliness  review_scores_location  calculated_host_listings_count  amenity_count      has_wifi   has_kitchen  has_air_conditioning  has_dedicated_workspace        has_tv   has_heating  has_refrigerator  has_microwave    has_washer  has_dishwasher  has_elevator       has_gym  availability_rate  price_per_person  price_per_bedroom     log_price  total_availability_score  review_score_avg  is_high_demand
count              20680.000000  20680.000000  20680.000000  20680.000000  20680.000000  20680.000000  20680.000000    20680.000000    2.068000e+04            20680.000000            2.068000e+04       20680.000000           20680.000000             

In [205]:
#as we can see that in the price, price_per_person, price_per_bedroom and minimum and maximum _nights
#has extreme outlier, as the mean and median are low but the max is very high
#therefore capping is required.
def cap_outliers(df, col):
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    print(f"{col}: capped at [{lower:.2f}, {upper:.2f}]")
    df[col] = df[col].clip(lower, upper)
    return df

# drop low variance columns first
housing = housing.drop(columns=['minimum_nights', 'minimum_nights_avg_ntm'])

# cap outliers
for col in ['price', 'price_per_person', 'price_per_bedroom',
            'maximum_nights', 'maximum_nights_avg_ntm',
            'number_of_reviews', 'number_of_reviews_ltm',
            'calculated_host_listings_count']:
    housing = cap_outliers(housing, col)

# verify
print("\nShape:", housing.shape)
print("Nulls:", housing.isnull().sum().sum())

price: capped at [-184.68, 558.00]
price_per_person: capped at [-48.80, 198.37]
price_per_bedroom: capped at [-140.36, 438.13]
maximum_nights: capped at [-787.50, 2272.50]
maximum_nights_avg_ntm: capped at [-1237.50, 2542.50]
number_of_reviews: capped at [-64.50, 107.50]
number_of_reviews_ltm: capped at [-4.50, 7.50]
calculated_host_listings_count: capped at [-17.00, 31.00]

Shape: (20680, 45)
Nulls: 0


In [206]:
print(housing.describe().to_string())

       hosts_time_as_host_years      latitude     longitude  accommodates     bathrooms      bedrooms         price  maximum_nights  maximum_nights_avg_ntm  number_of_reviews  number_of_reviews_ltm  estimated_occupancy_l365d  days_since_first_review  days_since_last_review   has_reviews  review_scores_cleanliness  review_scores_location  calculated_host_listings_count  amenity_count      has_wifi   has_kitchen  has_air_conditioning  has_dedicated_workspace        has_tv   has_heating  has_refrigerator  has_microwave    has_washer  has_dishwasher  has_elevator       has_gym  availability_rate  price_per_person  price_per_bedroom     log_price  total_availability_score  review_score_avg  is_high_demand
count              20680.000000  20680.000000  20680.000000  20680.000000  20680.000000  20680.000000  20680.000000    20680.000000            20680.000000       20680.000000           20680.000000               20680.000000             20680.000000            20680.000000  20680.000000   

In [207]:
print(housing.head(1000).to_string())

     hosts_time_as_host_years  host_is_superhost     neighbourhood_cleansed neighbourhood_group_cleansed   latitude  longitude                      property_type        room_type  accommodates  bathrooms  bedrooms    price  maximum_nights  maximum_nights_avg_ntm  has_availability  number_of_reviews  number_of_reviews_ltm  estimated_occupancy_l365d  days_since_first_review  days_since_last_review  has_reviews  review_scores_cleanliness  review_scores_location  calculated_host_listings_count  amenity_count  has_wifi  has_kitchen  has_air_conditioning  has_dedicated_workspace  has_tv  has_heating  has_refrigerator  has_microwave  has_washer  has_dishwasher  has_elevator  has_gym  availability_rate  price_per_person  price_per_bedroom  log_price host_experience_level  total_availability_score  review_score_avg  is_high_demand
0                        15.0               True               Williamsburg                     Brooklyn  40.709350 -73.953420                 Entire rental unit  Ent

In [208]:
housing['availability_rate'] = housing['availability_rate'] / 365

In [209]:
# drop is_high_demand
housing = housing.drop(columns=['is_high_demand'])

#as the model would learn if "high no of reviews = is high demand"

In [210]:
print(housing['number_of_reviews_ltm'].value_counts().head(10))


number_of_reviews_ltm
0.0    10903
7.5     3181
1.0     2540
2.0     1607
3.0     1074
4.0      634
5.0      409
6.0      202
7.0      130
Name: count, dtype: int64


In [211]:
print(housing.isnull().sum()[housing.isnull().sum() > 0])

Series([], dtype: int64)


In [212]:
#as we can see that capping created many Nan values so make some changes, like
#in host_exp level we can change the bin from 0 <-> -1
# and drop no_of_reviews_ltmm  as so many rows are 0 therefore now its not that important
housing = housing.drop(columns=['number_of_reviews_ltm'])
#i have updated the values in the host exp level so it might not appear in the prev cell now.

In [213]:
print("Nulls:", housing.isnull().sum().sum())
print("Shape:", housing.shape)

Nulls: 0
Shape: (20680, 43)


In [214]:
print(housing.duplicated().to_string())
#duplicates are created as the capped values gives NaN. Therefore we have to remove them.

0        False
1        False
2        False
3        False
4        False
5        False
6        False
7        False
8        False
9        False
10       False
11       False
12       False
13       False
14       False
15       False
16       False
17       False
18       False
19       False
20       False
21       False
22       False
23       False
24       False
25       False
26       False
27       False
28       False
29       False
30       False
31       False
32       False
33       False
34       False
35       False
36       False
37       False
38       False
39       False
40       False
41       False
42       False
43       False
44       False
45       False
46       False
47       False
48       False
49       False
50       False
51       False
52       False
53       False
54       False
55       False
56       False
57       False
58       False
59       False
60       False
61       False
62       False
63       False
64       False
65       False
66       F

In [215]:
print(housing.shape)
housing = housing.drop_duplicates()
print(housing.shape)

(20680, 43)
(20521, 43)


In [216]:
print(housing.head(10).to_string())

   hosts_time_as_host_years  host_is_superhost neighbourhood_cleansed neighbourhood_group_cleansed   latitude  longitude                property_type        room_type  accommodates  bathrooms  bedrooms   price  maximum_nights  maximum_nights_avg_ntm  has_availability  number_of_reviews  estimated_occupancy_l365d  days_since_first_review  days_since_last_review  has_reviews  review_scores_cleanliness  review_scores_location  calculated_host_listings_count  amenity_count  has_wifi  has_kitchen  has_air_conditioning  has_dedicated_workspace  has_tv  has_heating  has_refrigerator  has_microwave  has_washer  has_dishwasher  has_elevator  has_gym  availability_rate  price_per_person  price_per_bedroom  log_price host_experience_level  total_availability_score  review_score_avg
0                      15.0               True           Williamsburg                     Brooklyn  40.709350 -73.953420           Entire rental unit  Entire home/apt             3        1.0       2.0  117.27         

In [217]:
print(housing.columns.tolist())
print(housing.shape)
print(housing.isnull().sum().sum())

['hosts_time_as_host_years', 'host_is_superhost', 'neighbourhood_cleansed', 'neighbourhood_group_cleansed', 'latitude', 'longitude', 'property_type', 'room_type', 'accommodates', 'bathrooms', 'bedrooms', 'price', 'maximum_nights', 'maximum_nights_avg_ntm', 'has_availability', 'number_of_reviews', 'estimated_occupancy_l365d', 'days_since_first_review', 'days_since_last_review', 'has_reviews', 'review_scores_cleanliness', 'review_scores_location', 'calculated_host_listings_count', 'amenity_count', 'has_wifi', 'has_kitchen', 'has_air_conditioning', 'has_dedicated_workspace', 'has_tv', 'has_heating', 'has_refrigerator', 'has_microwave', 'has_washer', 'has_dishwasher', 'has_elevator', 'has_gym', 'availability_rate', 'price_per_person', 'price_per_bedroom', 'log_price', 'host_experience_level', 'total_availability_score', 'review_score_avg']
(20521, 43)
0


In [219]:
housing.to_csv(r'C:\Users\prith\Downloads\airbnb-data-pipeline\airbnb-data-pipeline\data\processed\featured_engineered_data.csv', index=False)